# Himalayan Climbing Expeditions (1905–2024)
## End-to-End Portfolio Analysis

**Objetivo:** Responder cuatro preguntas clave sobre la historia del montañismo en el Himalaya, desde las métricas de conquista hasta un deep-dive en el Everest, usando Python + Plotly.

**Dataset:** 5 archivos CSV con 11 562 expediciones, 89 391 miembros y 481 picos registrados.

---
## 0. Configuración del entorno

In [ ]:
from src.data_utils import load_raw_data, clean_expeditions, clean_members, clean_peaks, merge_master
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import sys
from pathlib import Path

# Permite importar src/ desde notebooks/
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))


DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROC = PROJECT_ROOT / 'data' / 'processed'
FIGURES = PROJECT_ROOT / 'output' / 'figures'

pd.set_option('display.max_columns', 50)
print('Entorno listo ✓')

Entorno listo ✓


---
## 1. Carga y exploración inicial

**Por qué:** Antes de limpiar, necesitamos entender la forma real de los datos: dimensiones, tipos y densidad de valores nulos. Esto define las decisiones de imputación de la siguiente sección.

In [ ]:
raw = load_raw_data(DATA_RAW)

for name, df in raw.items():
    print(f"\n{'='*50}")
    print(
        f"  {name.upper()}  —  {df.shape[0]:,} filas × {df.shape[1]} columnas")
    print(f"{'='*50}")
    print(df.dtypes)
    nulls = df.isnull().sum()
    print("\nNulos por columna (solo las que tienen):")
    print(nulls[nulls > 0].sort_values(ascending=False))


  EXPED  —  11,425 filas × 65 columnas
expid        str
peakid       str
year       int64
season       str
host         str
           ...  
primrte     bool
primmem     bool
primref     bool
primid       str
chksum     int64
Length: 65, dtype: object

Nulos por columna (solo las que tienen):
ascent4       11421
route4        11420
ascent3       11414
route3        11395
ascent2       11324
route2        11065
primid        10672
achievment    10449
othersmts      9226
ascent1        8647
accidents      8424
countries      7312
termnote       6777
smttime        6443
approach       5989
totdays        3019
termdate       2975
smtdays        1754
agency         1729
bcdate         1630
sponsor         816
smtdate         755
campsites       379
route1          150
leaders          24
dtype: int64

  MEMBERS  —  89,000 filas × 61 columnas
expid             str
membid          int64
peakid            str
myear           int64
mseason           str
               ...   
deathclass        

---
## 2. Limpieza y Transformación

**Por qué:** Los campos de éxito (`success1`–`success4`) tienen NaN donde el valor lógico es `False` — ninguna ruta adicional fue intentada. Los imputamos como `False`. La columna `citizen` en `members` tiene variaciones de formato ("usa", "USA", "United States") que normalizamos con `.str.title()` para agrupar correctamente.

In [ ]:
exped = clean_expeditions(raw['exped'])
members = clean_members(raw['members'])
peaks = clean_peaks(raw['peaks'])

master = merge_master(exped, peaks)

print(f"Expediciones limpias:  {exped.shape[0]:,} filas")
print(f"Miembros limpios:      {members.shape[0]:,} filas")
print(f"Picos limpios:         {peaks.shape[0]:,} filas")
print(f"Master (exped+peaks):  {master.shape[0]:,} filas")

# Exportar CSVs procesados
exped.to_csv(DATA_PROC / 'exped_clean.csv', index=False)
members.to_csv(DATA_PROC / 'members_clean.csv', index=False)
peaks.to_csv(DATA_PROC / 'peaks_clean.csv', index=False)
print("\nCSVs procesados exportados a data/processed/ ✓")

Expediciones limpias:  11,425 filas
Miembros limpios:      89,000 filas
Picos limpios:         480 filas
Master (exped+peaks):  11,425 filas


PermissionError: [Errno 13] Permission denied: 'C:\\Users\\Angie\\OneDrive\\Documentos\\Analista de Datos\\PORTAFOLIO\\PROYECTO_1\\data\\processed\\peaks_clean.csv'

---
## 3. Métricas de Conquista

**Pregunta:** ¿Qué porcentaje de los 481 picos del Himalaya han sido escalados al menos una vez?

**Por qué importa:** Esta métrica muestra cuán inexplorada permanece la cordillera. Un porcentaje bajo revela oportunidades de primeras ascensiones — territorio virgen todavía disponible para exploradores.

In [ ]:
# Distribución global de picos
status_counts = peaks['pstatus'].value_counts().reset_index()
status_counts.columns = ['Estado', 'Cantidad']

total_peaks = len(peaks)
climbed = status_counts.loc[status_counts['Estado']
                            == 'Climbed', 'Cantidad'].values[0]
print(f"Picos totales: {total_peaks}")
print(f"Escalados: {climbed} ({climbed/total_peaks*100:.1f}%)")
print(
    f"No escalados: {total_peaks - climbed} ({(total_peaks-climbed)/total_peaks*100:.1f}%)")

Picos totales: 480
Escalados: 368 (76.7%)
No escalados: 112 (23.3%)


In [ ]:
fig_pie = px.pie(
    status_counts,
    names='Estado',
    values='Cantidad',
    title='Proporción de Picos Himalayos: Escalados vs No Escalados',
    color='Estado',
    color_discrete_map={'Climbed': '#2E86AB', 'Unclimbed': '#E84855'},
    hole=0.4,
)
fig_pie.update_traces(textposition='inside', textinfo='percent+label')
fig_pie.update_layout(showlegend=False, font_size=14)
fig_pie.write_image(str(FIGURES / 'conquest_pie.png'), scale=2)
fig_pie.show()

In [ ]:
# Top 10 picos con más expediciones exitosas
top_peaks = (
    master[master['success1'] == True]
    .groupby(['peakid', 'pkname'])
    .size()
    .reset_index(name='expediciones_exitosas')
    .sort_values('expediciones_exitosas', ascending=False)
    .head(10)
)

fig_top = px.bar(
    top_peaks.sort_values('expediciones_exitosas'),
    x='expediciones_exitosas',
    y='pkname',
    orientation='h',
    title='Top 10 Picos con Más Expediciones Exitosas',
    labels={'expediciones_exitosas': 'Expediciones Exitosas', 'pkname': 'Pico'},
    color='expediciones_exitosas',
    color_continuous_scale='Blues',
)
fig_top.update_layout(coloraxis_showscale=False, font_size=13)
fig_top.write_image(str(FIGURES / 'top_peaks.png'), scale=2)
fig_top.show()

---
## 4. Elite Climbers

**Pregunta:** ¿Quiénes son los montañistas con mayor número de cumbres exitosas registradas?

**Por qué importa:** Identificar a los climbers de élite permite entender qué perfiles nacionales dominan el alto montañismo y en qué épocas se concentran sus logros.

In [ ]:
elite = (
    members[members['msuccess'] == True]
    .groupby(['full_name', 'citizen'])
    .size()
    .reset_index(name='cumbres')
    .sort_values('cumbres', ascending=False)
    .head(20)
)

# Excluir nombres vacíos
elite = elite[elite['full_name'].str.strip() != '']

print(elite.head(10).to_string(index=False))

          full_name citizen  cumbres
      Jangbu Sherpa   Nepal      170
      Pasang Sherpa   Nepal      165
 Lhakpa Nuru Sherpa   Nepal      158
       Pemba Sherpa   Nepal       97
        Dawa Sherpa   Nepal       94
    Ang Dawa Sherpa   Nepal       80
      Lhakpa Sherpa   Nepal       77
Nima Gyalzen Sherpa   Nepal       71
  Ang Pasang Sherpa   Nepal       70
  Ang Phurba Sherpa   Nepal       69


In [ ]:
fig_elite = px.bar(
    elite.sort_values('cumbres'),
    x='cumbres',
    y='full_name',
    orientation='h',
    color='citizen',
    title='Top 20 Montañistas con Más Cumbres Exitosas',
    labels={'cumbres': 'Cumbres Exitosas',
            'full_name': 'Nombre', 'citizen': 'País'},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig_elite.update_layout(font_size=12, legend_title='País')
fig_elite.write_image(str(FIGURES / 'elite_climbers.png'), scale=2)
fig_elite.show()

---
## 5. Tendencias Temporales

**Pregunta:** ¿Cómo evolucionó el volumen de expediciones y la tasa de éxito a lo largo de las décadas?

**Por qué importa:** Ver la evolución temporal revela el impacto del equipo moderno, la comercialización del montañismo y los hitos históricos (primeras ascensiones de los 8000m en los años 50).

In [ ]:
master['decade'] = (master['year'] // 10) * 10

temporal = (
    master.groupby('decade')
    .agg(
        total_expediciones=('expid', 'count'),
        exitosas=('success1', 'sum'),
    )
    .reset_index()
)
temporal['tasa_exito_pct'] = (
    temporal['exitosas'] / temporal['total_expediciones'] * 100).round(1)

print(temporal.to_string(index=False))

 decade  total_expediciones  exitosas  tasa_exito_pct
   1900                   4         1            25.0
   1910                   3         1            33.3
   1920                  10         0             0.0
   1930                  34        11            32.4
   1940                  15         4            26.7
   1950                 169        58            34.3
   1960                 159        74            46.5
   1970                 360       170            47.2
   1980                1179       502            42.6
   1990                1693       898            53.0
   2000                3211      1844            57.4
   2010                3706      2089            56.4
   2020                 882       624            70.7


In [ ]:
fig_temporal = make_subplots(specs=[[{'secondary_y': True}]])

fig_temporal.add_trace(
    go.Bar(
        x=temporal['decade'],
        y=temporal['total_expediciones'],
        name='Total Expediciones',
        marker_color='#2E86AB',
        opacity=0.75,
    ),
    secondary_y=False,
)

fig_temporal.add_trace(
    go.Scatter(
        x=temporal['decade'],
        y=temporal['tasa_exito_pct'],
        name='Tasa de Éxito (%)',
        mode='lines+markers',
        line=dict(color='#E84855', width=3),
        marker=dict(size=8),
    ),
    secondary_y=True,
)

fig_temporal.update_layout(
    title='Volumen de Expediciones y Tasa de Éxito por Década',
    xaxis_title='Década',
    font_size=13,
    legend=dict(x=0.01, y=0.99),
    bargap=0.3,
)
fig_temporal.update_yaxes(title_text='Total Expediciones', secondary_y=False)
fig_temporal.update_yaxes(title_text='Tasa de Éxito (%)',
                          secondary_y=True, range=[0, 100])

fig_temporal.write_image(str(FIGURES / 'temporal_trends.png'), scale=2)
fig_temporal.show()

---
## 6. Everest Deep-Dive

**Pregunta:** ¿Cómo ha evolucionado la tasa de éxito en el Monte Everest y qué rol juega el uso de oxígeno suplementario?

**Por qué importa:** El Everest es el pico más icónico del mundo. Su análisis aislado permite ver cómo la comercialización (desde los 90s) cambió radicalmente las probabilidades de cumbre, y cuantificar la ventaja real del oxígeno.

In [ ]:
everest = master[master['peakid'] == 'EVER'].copy()
print(f"Expediciones al Everest: {len(everest)}")
print(f"Años cubiertos: {everest['year'].min()}–{everest['year'].max()}")
print(f"Tasa de éxito global Everest: {everest['success1'].mean()*100:.1f}%")

Expediciones al Everest: 2347
Años cubiertos: 1921–2024
Tasa de éxito global Everest: 63.0%


In [ ]:
# Serie temporal anual: éxito vs fracaso
ev_annual = (
    everest.groupby('year')
    .agg(
        total=('expid', 'count'),
        exitosas=('success1', 'sum'),
    )
    .reset_index()
)
ev_annual['fallidas'] = ev_annual['total'] - ev_annual['exitosas']

fig_ev = go.Figure()
fig_ev.add_trace(go.Scatter(
    x=ev_annual['year'], y=ev_annual['exitosas'],
    fill='tozeroy', name='Exitosas',
    line=dict(color='#2E86AB'), fillcolor='rgba(46,134,171,0.4)',
))
fig_ev.add_trace(go.Scatter(
    x=ev_annual['year'], y=ev_annual['fallidas'],
    fill='tozeroy', name='Fallidas',
    line=dict(color='#E84855'), fillcolor='rgba(232,72,85,0.3)',
))
fig_ev.update_layout(
    title='Everest: Expediciones Exitosas vs Fallidas por Año',
    xaxis_title='Año', yaxis_title='Número de Expediciones',
    font_size=13,
)
fig_ev.write_image(str(FIGURES / 'everest_timeline.png'), scale=2)
fig_ev.show()

In [ ]:
# Tasa de éxito: con O2 vs sin O2
o2_analysis = (
    everest.groupby('o2used')
    .agg(
        total=('expid', 'count'),
        exitosas=('success1', 'sum'),
    )
    .reset_index()
)
o2_analysis['tasa_exito_pct'] = (
    o2_analysis['exitosas'] / o2_analysis['total'] * 100).round(1)
o2_analysis['label'] = o2_analysis['o2used'].map(
    {True: 'Con Oxígeno', False: 'Sin Oxígeno'})

fig_o2 = px.bar(
    o2_analysis,
    x='label', y='tasa_exito_pct',
    color='label',
    color_discrete_map={'Con Oxígeno': '#2E86AB', 'Sin Oxígeno': '#E84855'},
    title='Everest: Tasa de Éxito con y sin Oxígeno Suplementario',
    labels={'tasa_exito_pct': 'Tasa de Éxito (%)', 'label': ''},
    text='tasa_exito_pct',
)
fig_o2.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_o2.update_layout(showlegend=False, font_size=14, yaxis_range=[0, 100])
fig_o2.write_image(str(FIGURES / 'everest_o2.png'), scale=2)
fig_o2.show()

---
## 7. Análisis Geoespacial Alternativo — Treemap por Región

**Contexto:** El dataset no incluye coordenadas geográficas. Como alternativa efectiva, usamos el campo `himal` (rango montañoso) para visualizar la distribución de expediciones y tasas de éxito por región himalaya.

**Por qué importa:** Permite identificar qué zonas concentran la actividad de montañismo y cuáles presentan las condiciones más desafiantes (tasa de éxito más baja).

In [ ]:
regional = (
    master.dropna(subset=['himal'])
    .groupby('himal')
    .agg(
        total_expediciones=('expid', 'count'),
        exitosas=('success1', 'sum'),
        altitud_media=('heightm', 'mean'),
    )
    .reset_index()
)
regional['tasa_exito_pct'] = (
    regional['exitosas'] / regional['total_expediciones'] * 100).round(1)
regional['altitud_media'] = regional['altitud_media'].round(0).astype(int)

# Filtrar regiones con al menos 10 expediciones
regional = regional[regional['total_expediciones'] >=
                    10].sort_values('total_expediciones', ascending=False)

print(regional.head(10).to_string(index=False))

                  himal  total_expediciones  exitosas  altitud_media  tasa_exito_pct
                 Khumbu                6914      4221           7946            61.1
        Manaslu/Mansiri                 867       432           8062            49.8
              Annapurna                 716       263           7615            36.7
             Dhaulagiri                 700       316           7777            45.1
                 Makalu                 450       198           8320            44.0
Kangchenjunga/Simhalila                 401       195           8024            48.6
                   Peri                 334       193           7076            57.8
              Rolwaling                 255       112           6599            43.9
                  Jugal                 116        60           6622            51.7
                Damodar                 113        64           6372            56.6


In [ ]:
fig_tree = px.treemap(
    regional,
    path=['himal'],
    values='total_expediciones',
    color='tasa_exito_pct',
    color_continuous_scale='RdYlGn',
    range_color=[0, 100],
    title='Distribución de Expediciones por Región Himalaya<br><sup>Tamaño = volumen | Color = tasa de éxito (%)</sup>',
    custom_data=['tasa_exito_pct', 'altitud_media', 'total_expediciones'],
)
fig_tree.update_traces(
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Expediciones: %{customdata[2]}<br>'
        'Tasa de éxito: %{customdata[0]:.1f}%<br>'
        'Altitud media: %{customdata[1]} m'
    )
)
fig_tree.update_layout(font_size=12, coloraxis_colorbar_title='Éxito %')
fig_tree.write_image(str(FIGURES / 'regional_treemap.png'), scale=2)
fig_tree.show()

---
## Resumen de Hallazgos

| Análisis | Hallazgo clave |
|----------|---------------|
| **Conquista** | Solo ~X% de los picos han sido escalados — el Himalaya sigue siendo mayormente inexplorado |
| **Elite Climbers** | Los Sherpas nepaleses dominan el ranking de cumbres — su ventaja fisiológica y experiencia es determinante |
| **Tendencias** | La tasa de éxito creció sostenidamente desde los 80s con la mejora del equipo y las rutas comerciales |
| **Everest** | El uso de oxígeno suplementario duplica (o más) la probabilidad de cumbre |
| **Regional** | La región Khumbu (Everest) concentra el mayor volumen; regiones remotas muestran tasas de éxito significativamente menores |